## Former code

In [ ]:
import numpy as np
import h5py
from astropy.io import fits
from astropy.table import Table
import pandas as pd

In [ ]:
# This cell converts .fits data file from COSMOS-Web into .csv for later use

master_path= "C:\\Users\\usuario\\Documents\\TFG\\florah_training_SFR\\COSMOSWeb_mastercatalog_v1.fits"

def fits_to_pandas_clean(hdu):
    tbl = Table(hdu.data)
    names = [name for name in tbl.colnames if len(tbl[name].shape) <= 1]
    return tbl[names].to_pandas()

# Selecting only the extensions I will be using
with fits.open(master_path) as hdu:
    hdu.info()
    photom = fits_to_pandas_clean(hdu[1])
    leph   = fits_to_pandas_clean(hdu[2])
    cigale = fits_to_pandas_clean(hdu[4])
    morph  = fits_to_pandas_clean(hdu[5])
    bd     = fits_to_pandas_clean(hdu[6])

cosmos_cat = pd.concat([photom, leph, cigale, morph, bd], axis=1)
output_path = "C:\\Users\\usuario\\Documents\\TFG\\florah_training_SFR\\COSMOSWeb_laura.csv"
cosmos_cat.to_csv(output_path, index=False)


In [ ]:
data_path = "C:\\Users\\usuario\\Documents\\TFG\\florah_training_SFR\\"
cosmos_cat = pd.read_csv(data_path+"COSMOSWeb_laura.csv") # Data from COSMOS-WEB, converted from .fits to .csv in florah_eval_SFR.ipynb

I the next cell I will select the columns I want, from each extension selected before. This columns will be later saved in a new csv

In [ ]:
# Select the columns I want
columns = ['id', 'radius_sersic', 'ra', 'dec', 'sersic', 'sfr_med', 'mass_med', 'zpdf_med', 'sfr_inst', 'mass', 'morph_flag_f444w', 'b/t_f444w'  ]
cosmos_cat_filtered = cosmos_cat[columns]

# Save new csv
cosmos_cat_filtered.to_csv("C:\\Users\\usuario\\Documents\\TFG\\florah_training_SFR\\COSMOSWeb_Laura_filtered.csv", index=False)


## New code:

In [ ]:
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.table import Table
from astropy.cosmology import Planck13
import astropy.units as u

# 1. Converts .fits data file from COSMOS-Web into .csv for later use
master_path = "C:\\Users\\usuario\\Documents\\TFG\\florah_training_SFR\\COSMOSWeb_mastercatalog_v1.fits"

def fits_to_pandas_clean(hdu):
    tbl = Table(hdu.data)
    names = [name for name in tbl.colnames if len(tbl[name].shape) <= 1]
    return tbl[names].to_pandas()

# Selecting only the extensions I will be using
with fits.open(master_path) as hdu:
    photom = fits_to_pandas_clean(hdu[1])
    leph   = fits_to_pandas_clean(hdu[2])
    cigale = fits_to_pandas_clean(hdu[4])
    morph  = fits_to_pandas_clean(hdu[5])
    bd     = fits_to_pandas_clean(hdu[6])

cosmos_cat = pd.concat([photom, leph, cigale, morph, bd], axis=1)



# 2. Selection and initial cleaning of data
columns_map = {
    'id': 'id',
    'radius_sersic': 'radius_sersic',
    'ra': 'ra',
    'dec': 'dec',
    'sersic': 'sersic',
    'zpdf_med': 'zpdf_med',
    'sfr_inst': 'sfr_raw',
    'mass': 'mass_raw',
    'morph_flag_f444w': 'morphology',
    'b/t_f444w': 'bovert'
}

# Renombramos y filtramos para trabajar solo con lo necesario
cosmos_cat = cosmos_cat[list(columns_map.keys())].rename(columns=columns_map) 

# Convert every value to numeric, transforming errors into NaN
for col in cosmos_cat.columns:
    cosmos_cat[col] = pd.to_numeric(cosmos_cat[col], errors='coerce')



# 3. Logarithmic transformations and physical filters
# Filter impossible values before applying logarithms
cosmos_cat = cosmos_cat[(cosmos_cat['mass_raw'] > 0) & (cosmos_cat['sfr_raw'] > 0)]

cosmos_cat['mass_CIGALE'] = np.log10(cosmos_cat['mass_raw'])
cosmos_cat['sfr_CIGALE'] = np.log10(cosmos_cat['sfr_raw'])



# 4. Pre-calculation - A Phase
# This helps relieve later usage of data, so that astrophysical calculations, such as angular
# diamater distance, that will be once calculated and stored in the output file.
# NOTE: This could compromise RAM usage, but might speed up code
z_array = cosmos_cat['zpdf_med'].values
dist_mpc = Planck13.angular_diameter_distance(z_array).value # Resultado en Mpc


# Calculate physical radius in log10(kpc)
# Formula: Physical_radius = Angular_diameter * Angular_aperture(rad)
cosmos_cat['log_radius_kpc'] = np.log10(dist_mpc * np.deg2rad(cosmos_cat['radius_sersic']) * 1e3)



# 5. Store cleaned file
output_path = "C:\\Users\\usuario\\Documents\\TFG\\florah_training_SFR\\COSMOSWeb_Laura_processed.csv"
# Delete rows that contain NaNs in important columns
cosmos_cat.dropna(subset=['mass_CIGALE', 'sfr_CIGALE', 'zpdf_med', 'log_radius_kpc'], inplace=True)

cosmos_cat.to_csv(output_path, index=False)
print(f"Catálogo procesado guardado con {len(cosmos_cat)} filas.")

In [ ]:
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.table import Table
from astropy.cosmology import Planck13
import astropy.units as u

# 1. Converts .fits data file from COSMOS-Web into .csv for later use
master_path = "C:\\Users\\usuario\\Documents\\TFG\\florah_training_SFR\\COSMOSWeb_mastercatalog_v1.fits"

def fits_to_pandas_clean(hdu):
    tbl = Table(hdu.data)
    names = [name for name in tbl.colnames if len(tbl[name].shape) <= 1]
    return tbl[names].to_pandas()

# Selecting only the extensions I will be using
with fits.open(master_path) as hdu:
    photom = fits_to_pandas_clean(hdu[1])
    morph  = fits_to_pandas_clean(hdu[5])

cosmos_cat = pd.concat([photom, morph], axis=1)


# 2. Selection and initial cleaning of data
columns_map = {
    'id': 'id',
    'morph_flag_f444w': 'morphology',
    'delta_f444w': 'delta_f444w'
}

# Renombramos y filtramos para trabajar solo con lo necesario
cosmos_cat = cosmos_cat[list(columns_map.keys())].rename(columns=columns_map) 


obj_id = 737995
# Iterate through the indices of node_features to find the matching ID
for i in range(len(cosmos_cat['id'])):        
    # Safest way to check if ID is in the numpy array
    if cosmos_cat['id'][i] == obj_id:
        index = i
        break
    else: 
        continue


# Convert every value to numeric, transforming errors into NaN
for col in cosmos_cat.columns:
    cosmos_cat[col] = pd.to_numeric(cosmos_cat[col], errors='coerce')

print(cosmos_cat['id'][index])
print(cosmos_cat['morphology'][index])
print(cosmos_cat['delta_f444w'][index])

## Creating another .csv for plotting Cigale's SFHs

In [ ]:
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.table import Table
from astropy.cosmology import Planck13
import astropy.units as u


# 1. Converts .fits data file from COSMOS-Web into .csv for later use
master_path = "C:\\Users\\usuario\\Documents\\TFG\\florah_training_SFR\\COSMOSWeb_mastercatalog_v1.fits"

def fits_to_pandas_clean(hdu):
    tbl = Table(hdu.data)
    names = [name for name in tbl.colnames if len(tbl[name].shape) <= 1]
    return tbl[names].to_pandas()

# Selecting only the extensions I will be using
with fits.open(master_path) as hdu:

    photom = fits_to_pandas_clean(hdu[1])
    leph   = fits_to_pandas_clean(hdu[2])
    cigale = fits_to_pandas_clean(hdu[4])



cosmos_cat = pd.concat([photom, leph, cigale], axis=1)


# 2. Selection and initial cleaning of data
columns_map = {
    'id': 'id',

    'zfinal': 'zfinal',
    'zpdf_med': 'zpdf_med',
    'mabs_nuv': 'mabs_nuv',
    'mabs_r': 'mabs_r',
    'mabs_j': 'mabs_j',

    'sfh_sfr_bin1': 'sfh_sfr_bin1',
    'sfh_sfr_bin2': 'sfh_sfr_bin2',
    'sfh_sfr_bin3': 'sfh_sfr_bin3',
    'sfh_sfr_bin4': 'sfh_sfr_bin4',
    'sfh_sfr_bin5': 'sfh_sfr_bin5',
    'sfh_sfr_bin6': 'sfh_sfr_bin6',
    'sfh_sfr_bin7': 'sfh_sfr_bin7',
    'sfh_sfr_bin8': 'sfh_sfr_bin8',
    'sfh_sfr_bin9': 'sfh_sfr_bin9',    
    
    'sfh_sfr_bin1_err': 'sfh_sfr_bin1_err',
    'sfh_sfr_bin2_err': 'sfh_sfr_bin2_err',
    'sfh_sfr_bin3_err': 'sfh_sfr_bin3_err',
    'sfh_sfr_bin4_err': 'sfh_sfr_bin4_err',
    'sfh_sfr_bin5_err': 'sfh_sfr_bin5_err',
    'sfh_sfr_bin6_err': 'sfh_sfr_bin6_err',
    'sfh_sfr_bin7_err': 'sfh_sfr_bin7_err',
    'sfh_sfr_bin8_err': 'sfh_sfr_bin8_err',
    'sfh_sfr_bin9_err': 'sfh_sfr_bin9_err',    
    
    'sfh_time_bin1': 'sfh_time_bin1',
    'sfh_time_bin2': 'sfh_time_bin2',
    'sfh_time_bin3': 'sfh_time_bin3',
    'sfh_time_bin4': 'sfh_time_bin4',
    'sfh_time_bin5': 'sfh_time_bin5',
    'sfh_time_bin6': 'sfh_time_bin6',
    'sfh_time_bin7': 'sfh_time_bin7',
    'sfh_time_bin8': 'sfh_time_bin8',
    'sfh_time_bin9': 'sfh_time_bin9',    
    
    'sfh_time_bin1_err': 'sfh_time_bin1_err',
    'sfh_time_bin2_err': 'sfh_time_bin2_err',
    'sfh_time_bin3_err': 'sfh_time_bin3_err',
    'sfh_time_bin4_err': 'sfh_time_bin4_err',
    'sfh_time_bin5_err': 'sfh_time_bin5_err',
    'sfh_time_bin6_err': 'sfh_time_bin6_err',
    'sfh_time_bin7_err': 'sfh_time_bin7_err',
    'sfh_time_bin8_err': 'sfh_time_bin8_err',
    'sfh_time_bin9_err': 'sfh_time_bin9_err',

    'sfh_integrated': 'sfh_integrated',
    'sfh_integrated_err': 'sfh_integrated_err'
}

# 3. Take only the columns we are interested in
cosmos_cat = cosmos_cat[list(columns_map.keys())]

In [ ]:
"""
# 4. Pre-calculation -
# Denormalize sfh and error. Also convert time to the right units
for i in range(1,10):
    # We denormalize sfr firstly
    cosmos_cat['sfh_sfr_bin'+str(i)] = cosmos_cat['sfh_sfr_bin'+str(i)] * cosmos_cat['sfh_integrated']
    cosmos_cat['sfh_sfr_bin'+str(i)] = np.log10(cosmos_cat['sfh_sfr_bin'+str(i)])

    # We calculate true error for sfr by propagating.
    cosmos_cat['sfh_sfr_bin'+str(i)+'_prop_err'] = cosmos_cat['sfh_sfr_bin'+str(i)+'_err'] * cosmos_cat['sfh_integrated'] + cosmos_cat['sfh_sfr_bin'+str(i)] * cosmos_cat['sfh_integrated_err']
    cosmos_cat['sfh_sfr_bin'+str(i)+'_prop_err'] = np.log10(cosmos_cat['sfh_sfr_bin'+str(i)+'_err'])
    # We will also store error in each bin just in case.
    cosmos_cat['sfh_sfr_bin'+str(i)+'_err'] = np.log10(cosmos_cat['sfh_sfr_bin'+str(i)+'_err'] * cosmos_cat['sfh_integrated_err'])
    

    # We clean time data -> From Myr to Gyr
    cosmos_cat['sfh_time_bin'+str(i)] = cosmos_cat['sfh_time_bin'+str(i)] * 10**(-3)
    cosmos_cat['sfh_time_bin'+str(i)+'_err'] = cosmos_cat['sfh_time_bin'+str(i)+'_err'] * 10**(-3)
"""

In [ ]:
# 5. Store cleaned file - One file per galaxy
output_path = r"C:\Users\usuario\Documents\TFG\likelihood_COSMOS_SFR\id_sfh\\"
files = []

# We'll process one file per galaxy, containing the data in all of the 9 bins for that galaxy.
for j in range(len(cosmos_cat)):
    all_sfr = []
    all_time = []
    all_sfr_err = []
    all_time_err = []
    all_sfh_int = []
    all_sfh_int_err = []
    all_id = []
    zpdf_med =[]
    zfinal = []
    mabs_nuv = []
    mabs_r = []
    mabs_j = []

    
    id_num = str(cosmos_cat['id'][j-1])

    # Find data for j galaxy in all 9 bin files and append value to lists
    for q in range(1,10):
        id_num = str(cosmos_cat['id'][j-1])
        all_id.append(id_num)      

        sfr_in_bin = cosmos_cat['sfh_sfr_bin'+str(q)][j-1]
        all_sfr.append(sfr_in_bin)        
        
        sfr_err_in_bin = cosmos_cat['sfh_sfr_bin'+str(q)+'_err'][j-1]
        all_sfr_err.append(sfr_err_in_bin)        
        
        time_in_bin = cosmos_cat['sfh_time_bin'+str(q)][j-1]
        all_time.append(time_in_bin)

        time_err_in_bin = cosmos_cat['sfh_time_bin'+str(q)+'_err'][j-1]
        all_time_err.append(time_err_in_bin) 

        sfh_int = cosmos_cat['sfh_integrated'][j-1]
        all_sfh_int.append(sfh_int) 

        sfh_int_err = cosmos_cat['sfh_integrated_err'][j-1]
        all_sfh_int_err.append(sfh_int_err)

        zpdf_med_value = cosmos_cat['zpdf_med'][j-1]
        zpdf_med.append(zpdf_med_value)

        zfinal_value = cosmos_cat['zfinal'][j-1]
        zfinal.append(zfinal_value)

        mabs_nuv_value = cosmos_cat['mabs_nuv'][j-1]
        mabs_nuv.append(mabs_nuv_value)

        mabs_r_value = cosmos_cat['mabs_r'][j-1]
        mabs_r.append(mabs_r_value)

        mabs_j_value = cosmos_cat['mabs_j'][j-1]
        mabs_j.append(mabs_j_value)




    # Combine your lists into a dictionary, then convert to a Pandas DataFrame
    galaxy_data = pd.DataFrame({
        'id': id_num, # Unique ID of the source
        'time': all_time, # lbt [Myr]
        'time_err': all_time_err, # Error in lbt [Myr]
        'sfr': all_sfr, # Normalized SFR [1/yr]
        'sfr_err': all_sfr_err, # Normalized error in SFR [1/yr]
        'sfh_int': all_sfh_int, # Total integrated star formation history [M_sol]
        'sfh_int_err': all_sfh_int_err, # Error in sfh_integrated [M_sol]
        'zpdf_med': zpdf_med,
        'zfinal': zfinal,
        'mabs_nuv': mabs_nuv,
        'mabs_j': mabs_j,
        'mabs_r': mabs_r
    })

    # Clean NaNs
    galaxy_data = galaxy_data.dropna(axis=0, ignore_index=True)

    # Define the file name
    file_name = f"CIGALE_{id_num}_sfh.csv"
    files.append(id_num)

    # Export the DataFrame to a CSV
    #galaxy_data.to_csv(output_path+file_name, index=False)


print(len(files))

## If we only want to update a specific file, run:

In [ ]:
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.table import Table
from astropy.cosmology import Planck13
import astropy.units as u
from tqdm import tqdm


# 1. Converts .fits data file from COSMOS-Web into .csv for later use
master_path = "C:\\Users\\usuario\\Documents\\TFG\\florah_training_SFR\\COSMOSWeb_mastercatalog_v1.fits"

def fits_to_pandas_clean(hdu):
    tbl = Table(hdu.data)
    names = [name for name in tbl.colnames if len(tbl[name].shape) <= 1]
    return tbl[names].to_pandas()

# Selecting only the extensions I will be using
with fits.open(master_path) as hdu:

    photom = fits_to_pandas_clean(hdu[1])
    leph   = fits_to_pandas_clean(hdu[2])
    cigale = fits_to_pandas_clean(hdu[4])



cosmos_cat = pd.concat([photom, leph, cigale], axis=1)


# 2. Selection and initial cleaning of data
columns_map = {
    'id': 'id',

    'zfinal': 'zfinal',
    'zpdf_med': 'zpdf_med',
    'mabs_nuv': 'mabs_nuv',
    'mabs_r': 'mabs_r',
    'mabs_j': 'mabs_j',

    'sfh_sfr_bin1': 'sfh_sfr_bin1',
    'sfh_sfr_bin2': 'sfh_sfr_bin2',
    'sfh_sfr_bin3': 'sfh_sfr_bin3',
    'sfh_sfr_bin4': 'sfh_sfr_bin4',
    'sfh_sfr_bin5': 'sfh_sfr_bin5',
    'sfh_sfr_bin6': 'sfh_sfr_bin6',
    'sfh_sfr_bin7': 'sfh_sfr_bin7',
    'sfh_sfr_bin8': 'sfh_sfr_bin8',
    'sfh_sfr_bin9': 'sfh_sfr_bin9',    
    
    'sfh_sfr_bin1_err': 'sfh_sfr_bin1_err',
    'sfh_sfr_bin2_err': 'sfh_sfr_bin2_err',
    'sfh_sfr_bin3_err': 'sfh_sfr_bin3_err',
    'sfh_sfr_bin4_err': 'sfh_sfr_bin4_err',
    'sfh_sfr_bin5_err': 'sfh_sfr_bin5_err',
    'sfh_sfr_bin6_err': 'sfh_sfr_bin6_err',
    'sfh_sfr_bin7_err': 'sfh_sfr_bin7_err',
    'sfh_sfr_bin8_err': 'sfh_sfr_bin8_err',
    'sfh_sfr_bin9_err': 'sfh_sfr_bin9_err',    
    
    'sfh_time_bin1': 'sfh_time_bin1',
    'sfh_time_bin2': 'sfh_time_bin2',
    'sfh_time_bin3': 'sfh_time_bin3',
    'sfh_time_bin4': 'sfh_time_bin4',
    'sfh_time_bin5': 'sfh_time_bin5',
    'sfh_time_bin6': 'sfh_time_bin6',
    'sfh_time_bin7': 'sfh_time_bin7',
    'sfh_time_bin8': 'sfh_time_bin8',
    'sfh_time_bin9': 'sfh_time_bin9',    
    
    'sfh_time_bin1_err': 'sfh_time_bin1_err',
    'sfh_time_bin2_err': 'sfh_time_bin2_err',
    'sfh_time_bin3_err': 'sfh_time_bin3_err',
    'sfh_time_bin4_err': 'sfh_time_bin4_err',
    'sfh_time_bin5_err': 'sfh_time_bin5_err',
    'sfh_time_bin6_err': 'sfh_time_bin6_err',
    'sfh_time_bin7_err': 'sfh_time_bin7_err',
    'sfh_time_bin8_err': 'sfh_time_bin8_err',
    'sfh_time_bin9_err': 'sfh_time_bin9_err',

    'sfh_integrated': 'sfh_integrated',
    'sfh_integrated_err': 'sfh_integrated_err'
}

# 3. Take only the columns we are interested in
cosmos_cat = cosmos_cat[list(columns_map.keys())]


In [ ]:
# 5. Store cleaned file - One file per galaxy
output_path = r"C:\Users\usuario\Documents\TFG\likelihood_COSMOS_SFR\id_sfh_3\\"
#id_to_plot = [508710, 525621, 52999, 691283, 233804, 755470, 240207, 666286, 734273, 175080, 734956, 713318, 315646, 402560, 714163, 393092, 587072, 21590, 391150, 144660, 449124, 515418, 518723, 22471, 386299, 585912, 131454, 51727, 676536, 131451, 733398, 361, 588848, 330660, 623991, 328465, 165836, 195598, 294261, 90335, 623707, 105316, 431840, 139731, 310896, 649302, 728441, 131751, 670037, 160309, 446131, 137270, 510377, 135134, 579710, 330386, 130567, 514267, 717051, 118850, 190051, 330499, 500470, 715240, 401378, 400210, 636148]
id_to_plot = [450847, 597391, 123102, 768357, 131823, 648996, 17550, 758802, 513732, 179675, 336825, 518203, 63681, 6133, 316563, 522211, 167146, 428119, 714273, 6607, 314349, 543284, 178413, 525843, 130405, 505728, 690100, 206945, 247227, 688347, 518693, 212603, 483413, 200733, 691311, 132889, 137269, 444308, 729411, 295693, 198110, 443615, 60018, 678159, 430462, 675525, 510905, 19938, 79720, 320911, 424984, 176477, 717052, 522362, 88690, 316667, 763175, 118510, 530819, 599866, 278446, 736714, 526734, 60214, 530766, 515845, 624418, 583536, 627491, 600730, 201909, 446760, 575504, 677831, 314677, 140559, 479270, 159139, 484303, 513042, 676074, 749487, 501819, 315704, 170001, 400469, 62367, 253275, 5018, 5029, 707876, 295209, 373637, 751746, 22935, 365327, 606216, 117978, 476148, 585924, 557554, 6635, 319394, 351702, 236147, 619914, 27203, 368104, 22262, 517310, 249633, 93291, 508653, 711119, 677140, 552574, 122592, 230897, 428588, 514269, 358274, 158183, 672083, 236951, 598711, 232600, 622166, 671965, 728575, 4767, 316452, 427341, 389029, 479722, 622488, 683837, 198805, 60304, 566700, 357651, 445345, 132689, 11197, 623710, 128484, 121385, 353804, 401177, 214342, 157160, 399946, 2852, 157604, 478599, 601193, 646085, 272585, 282681, 438833, 526294, 335214, 625714, 202392, 22172, 118830, 86230, 751065, 669292, 126196, 484692, 366028, 527307, 525247, 650073, 188645, 547376, 318003, 320586, 131453, 290814, 282773, 44653, 97758, 450601, 403639, 15843, 6725, 690999, 57987, 586732, 765878, 588749, 361532, 474666, 160227, 635920, 508710, 524573, 650176, 330502, 293409, 292069, 619815, 12320, 252404, 722879, 436823, 132090, 520550, 439447, 628668, 440359, 6592, 328122, 515517, 601287, 730983, 722915, 13426, 7629, 443298, 675701, 135240, 718935, 449601, 441468, 412205, 686618, 514733, 445698, 602427, 514585, 91864, 277036, 755470, 732971, 59590, 722503, 547596, 207452, 394239, 441064, 368779, 480635, 511120, 135355, 445086, 764977, 556903, 390890, 349450, 163382, 450032, 553823, 83131, 253679, 406219, 5292, 356272, 518204, 38571, 726580, 525621, 168897, 135472, 8953, 20462, 687514, 663430, 408058, 296515, 26544, 83565, 433433, 559258, 411027, 596571, 159794, 553963, 441469, 15154, 81024, 445386, 83341, 22879, 752281, 649816, 412883, 523879, 435968, 127604, 637950, 577116, 389461, 526474, 714558, 685484, 443150, 238968, 84736, 728526, 629960, 722988, 719903, 487887, 144974, 475596, 606285, 567237, 713653, 352717, 471527, 282901, 517451, 444894, 242235, 280515, 44141, 91750, 237756, 758044, 515285, 53255, 511201, 208318, 61155, 279656, 349447, 171218, 469314, 15727, 905, 638881, 439242, 102338, 398888, 434762, 566328, 487358, 105439, 506812, 511073, 734638, 397793, 620071, 5156, 432337, 43786, 364402, 522627, 529804, 517872, 664512, 4834, 24433, 5720, 125201, 523313, 143888, 5452, 360214, 6907, 329416, 176740, 723610, 643144, 442633, 101556, 505907, 678741, 605790, 6638, 244148, 513982, 175964, 51431, 144070, 284414, 312178, 293189, 278003, 136534, 208895, 635837, 605827, 208887, 12267, 711290, 281506, 209948, 144975, 91174, 560340, 443613, 292611, 19720, 140504, 468536, 273776, 170752, 286022, 176609, 631469, 521762, 389140, 678717, 351668, 140103, 242012, 759573, 443211, 291438, 631865, 734397, 582020, 199944, 525156, 647050, 42784, 730420, 431323, 55518, 717144, 373525, 239414, 364202, 372870, 86231, 330293, 61042, 477043, 83239, 26766, 410951, 7677, 140890, 432687, 488480, 50364, 442960, 600517, 276701, 389028, 237515, 476226, 85637, 402573, 432600, 556549, 14071, 40487, 355418, 203878, 371009, 15360, 82942, 526526, 501932, 53873, 515444, 712161, 517975, 692309, 513511, 513986, 548683, 2693, 577127, 450958, 57977, 275954, 481816, 760746, 641929, 595092, 177856, 585671, 443254, 424780, 754331, 602215, 707172, 394255, 735483, 577052, 512858, 623709, 328102, 60114, 716182, 195040, 469151, 399872, 83970, 366522, 647164, 53391, 235223, 705865, 551997, 2187, 24592, 229787, 761264, 605835, 329461, 432978, 26896, 332763, 401524, 130439, 255125, 605487, 389798, 179341, 593167, 161337, 410834, 59176, 430196, 714268, 514524, 45487, 565550, 408622, 756981, 322083, 400689, 235031, 6326, 595748, 666286, 585670, 667538, 760206, 717864, 451047, 718362, 175080, 98008, 749264, 322933, 23572, 63626, 401337, 597126, 641871, 522961, 544486, 587237, 513044, 409530, 217115, 433292, 121798, 287627, 604188, 128614, 412130, 519050, 324015, 598121, 124998, 441556, 600770, 167195, 586629, 679507, 237768, 204787, 734273, 360, 444478, 238758, 129909, 726581, 85539, 517745, 724122, 233444, 401793, 323875, 171685, 131452, 319158, 288691, 322147, 195041, 354215, 733398, 715240, 131451, 676536, 393092, 449124, 401378, 587072, 431840, 160309, 391150, 361, 714163, 195598, 137270, 579710, 728441, 118850, 518723, 330660, 636148, 90335, 623991, 315646, 515418, 734956, 22471, 21590, 165836, 400210, 131751, 130567, 500470, 105316, 623707, 717051, 588848, 713318, 51727, 585912, 649302, 328465, 310896, 294261, 670037, 330386, 446131, 386299, 190051, 514267, 139731, 402560, 510377, 131454, 135134, 330499, 144660]

# We'll process one file per galaxy, containing the data in all of the 9 bins for that galaxy.
for j in tqdm(id_to_plot):
    all_sfr = []
    all_time = []
    all_sfr_err = []
    all_time_err = []
    all_sfh_int = []
    all_sfh_int_err = []
    all_id = []
    zpdf_med =[]
    zfinal = []
    mabs_nuv = []
    mabs_r = []
    mabs_j = []


    # Find data for j galaxy in all 9 bin files and append value to lists
    for q in range(1,10):
        id_num = str(cosmos_cat['id'][j])
        all_id.append(id_num)      

        sfr_in_bin = cosmos_cat['sfh_sfr_bin'+str(q)][j]
        all_sfr.append(sfr_in_bin)        
        
        sfr_err_in_bin = cosmos_cat['sfh_sfr_bin'+str(q)+'_err'][j]
        all_sfr_err.append(sfr_err_in_bin)        
        
        time_in_bin = cosmos_cat['sfh_time_bin'+str(q)][j]
        all_time.append(time_in_bin)

        time_err_in_bin = cosmos_cat['sfh_time_bin'+str(q)+'_err'][j]
        all_time_err.append(time_err_in_bin) 

        sfh_int = cosmos_cat['sfh_integrated'][j]
        all_sfh_int.append(sfh_int) 

        sfh_int_err = cosmos_cat['sfh_integrated_err'][j]
        all_sfh_int_err.append(sfh_int_err)

        zpdf_med_value = cosmos_cat['zpdf_med'][j]
        zpdf_med.append(zpdf_med_value)

        zfinal_value = cosmos_cat['zfinal'][j]
        zfinal.append(zfinal_value)

        mabs_nuv_value = cosmos_cat['mabs_nuv'][j]
        mabs_nuv.append(mabs_nuv_value)

        mabs_r_value = cosmos_cat['mabs_r'][j]
        mabs_r.append(mabs_r_value)

        mabs_j_value = cosmos_cat['mabs_j'][j]
        mabs_j.append(mabs_j_value)




    # Combine your lists into a dictionary, then convert to a Pandas DataFrame
    galaxy_data = pd.DataFrame({
        'id': cosmos_cat['id'][j], # Unique ID of the source
        'time': all_time, # lbt [Myr]
        'time_err': all_time_err, # Error in lbt [Myr]
        'sfr': all_sfr, # Normalized SFR [1/yr]
        'sfr_err': all_sfr_err, # Normalized error in SFR [1/yr]
        'sfh_int': all_sfh_int, # Total integrated star formation history [M_sol]
        'sfh_int_err': all_sfh_int_err, # Error in sfh_integrated [M_sol]
        'zpdf_med': zpdf_med,
        'zfinal': zfinal,
        'mabs_nuv': mabs_nuv,
        'mabs_j': mabs_j,
        'mabs_r': mabs_r
    })

    # Clean NaNs
    galaxy_data = galaxy_data.dropna(axis=0, ignore_index=True)

    # Define the file name
    file_name = f"CIGALE_{id_num}_sfh.csv"

    # Export the DataFrame to a CSV
    galaxy_data.to_csv(output_path+file_name, index=False)


# Archivo definitivo - CIGALE data y COSMOS data

In [1]:
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.table import Table
from astropy.cosmology import Planck13
import astropy.units as u

# 1. Converts .fits data file from COSMOS-Web into .csv for later use
master_path = "C:\\Users\\usuario\\Documents\\TFG\\florah_training_SFR\\COSMOSWeb_mastercatalog_v1.fits"

def fits_to_pandas_clean(hdu):
    tbl = Table(hdu.data)
    names = [name for name in tbl.colnames if len(tbl[name].shape) <= 1]
    return tbl[names].to_pandas()

# Selecting only the extensions I will be using
with fits.open(master_path) as hdu:
    photom = fits_to_pandas_clean(hdu[1])
    leph   = fits_to_pandas_clean(hdu[2])
    cigale = fits_to_pandas_clean(hdu[4])
    morph  = fits_to_pandas_clean(hdu[5])
    bd     = fits_to_pandas_clean(hdu[6])

cosmos_cat = pd.concat([photom, leph, cigale, morph, bd], axis=1)



# 2. Selection and initial cleaning of columns - Cambiamos los nombres
columns_map = {
    'id': 'id',
    'radius_sersic': 'radius_sersic',
    'ra': 'ra',
    'dec': 'dec',
    'sersic': 'sersic',
    'zpdf_med': 'zpdf_med',
    'sfr_inst': 'sfr_raw',
    'mass': 'mass_raw',
    'morph_flag_f444w': 'morphology',
    'b/t_f444w': 'bovert',


    'zfinal': 'zfinal',
    'mabs_nuv': 'mabs_nuv',
    'mabs_r': 'mabs_r',
    'mabs_j': 'mabs_j',

    'sfh_sfr_bin1': 'sfh_sfr_bin1',
    'sfh_sfr_bin2': 'sfh_sfr_bin2',
    'sfh_sfr_bin3': 'sfh_sfr_bin3',
    'sfh_sfr_bin4': 'sfh_sfr_bin4',
    'sfh_sfr_bin5': 'sfh_sfr_bin5',
    'sfh_sfr_bin6': 'sfh_sfr_bin6',
    'sfh_sfr_bin7': 'sfh_sfr_bin7',
    'sfh_sfr_bin8': 'sfh_sfr_bin8',
    'sfh_sfr_bin9': 'sfh_sfr_bin9',    
    
    'sfh_sfr_bin1_err': 'sfh_sfr_bin1_err',
    'sfh_sfr_bin2_err': 'sfh_sfr_bin2_err',
    'sfh_sfr_bin3_err': 'sfh_sfr_bin3_err',
    'sfh_sfr_bin4_err': 'sfh_sfr_bin4_err',
    'sfh_sfr_bin5_err': 'sfh_sfr_bin5_err',
    'sfh_sfr_bin6_err': 'sfh_sfr_bin6_err',
    'sfh_sfr_bin7_err': 'sfh_sfr_bin7_err',
    'sfh_sfr_bin8_err': 'sfh_sfr_bin8_err',
    'sfh_sfr_bin9_err': 'sfh_sfr_bin9_err',    
    
    'sfh_time_bin1': 'sfh_time_bin1',
    'sfh_time_bin2': 'sfh_time_bin2',
    'sfh_time_bin3': 'sfh_time_bin3',
    'sfh_time_bin4': 'sfh_time_bin4',
    'sfh_time_bin5': 'sfh_time_bin5',
    'sfh_time_bin6': 'sfh_time_bin6',
    'sfh_time_bin7': 'sfh_time_bin7',
    'sfh_time_bin8': 'sfh_time_bin8',
    'sfh_time_bin9': 'sfh_time_bin9',    
    
    'sfh_time_bin1_err': 'sfh_time_bin1_err',
    'sfh_time_bin2_err': 'sfh_time_bin2_err',
    'sfh_time_bin3_err': 'sfh_time_bin3_err',
    'sfh_time_bin4_err': 'sfh_time_bin4_err',
    'sfh_time_bin5_err': 'sfh_time_bin5_err',
    'sfh_time_bin6_err': 'sfh_time_bin6_err',
    'sfh_time_bin7_err': 'sfh_time_bin7_err',
    'sfh_time_bin8_err': 'sfh_time_bin8_err',
    'sfh_time_bin9_err': 'sfh_time_bin9_err',

    'sfh_integrated': 'sfh_integrated',
    'sfh_integrated_err': 'sfh_integrated_err'

}

# Renombramos y filtramos para trabajar solo con lo necesario
cosmos_cat = cosmos_cat[list(columns_map.keys())].rename(columns=columns_map) 

# Convert every value to numeric, transforming errors into NaN
for col in cosmos_cat.columns:
    cosmos_cat[col] = pd.to_numeric(cosmos_cat[col], errors='coerce')


# 3. Logarithmic transformations and physical filters
# Filter impossible values before applying logarithms
cosmos_cat = cosmos_cat[(cosmos_cat['mass_raw'] > 0) & (cosmos_cat['sfr_raw'] > 0)]

cosmos_cat['mass_CIGALE'] = np.log10(cosmos_cat['mass_raw'])
cosmos_cat['sfr_CIGALE'] = np.log10(cosmos_cat['sfr_raw'])


# 4. Pre-calculation
# This helps relieve later usage of data, so that astrophysical calculations, such as angular
# diamater distance, that will be once calculated and stored in the output file.
# NOTE: This could compromise RAM usage, but might speed up code
z_array = cosmos_cat['zpdf_med'].values
dist_mpc = Planck13.angular_diameter_distance(z_array).value # Resultado en Mpc


# Calculate physical radius in log10(kpc)
# Formula: Physical_radius = Angular_diameter * Angular_aperture(rad)
cosmos_cat['log_radius_kpc'] = np.log10(dist_mpc * np.deg2rad(cosmos_cat['radius_sersic']) * 1e3)

# Delete rows that contain NaNs in important columns
cosmos_cat.dropna(subset=['mass_CIGALE', 'sfr_CIGALE', 'zpdf_med', 'log_radius_kpc'], inplace=True)


all_sfr_total = []
all_time_total = []
all_sfr_err_total = []
all_time_err_total = []
all_sfh_int_total = []
all_sfh_int_err_total = []


sfr_cols = [f'sfh_sfr_bin{q}' for q in range(1, 10)]
sfr_err_cols = [f'sfh_sfr_bin{q}_err' for q in range(1, 10)]
time_cols = [f'sfh_time_bin{q}' for q in range(1, 10)]
time_err_cols = [f'sfh_time_bin{q}_err' for q in range(1, 10)]

# 2. Extract the data as a list of NumPy arrays instantly
all_sfr_total = list(cosmos_cat[sfr_cols].astype(float).values)
all_sfr_err_total = list(cosmos_cat[sfr_err_cols].astype(float).values)
all_time_total = list(cosmos_cat[time_cols].astype(float).values)
all_time_err_total = list(cosmos_cat[time_err_cols].astype(float).values)

# 3. Replicate logic for integrated values using vectorization instead of a Python loop
# np.repeat creates a 2D array of shape (N, 9), and list() splits it into 1D arrays per row
all_sfh_int_total = list(np.repeat(cosmos_cat['sfh_integrated'].astype(float).values[:, None], 9, axis=1))
all_sfh_int_err_total = list(np.repeat(cosmos_cat['sfh_integrated_err'].astype(float).values[:, None], 9, axis=1))



# Combine your lists into a dictionary, then convert to a Pandas DataFrame
galaxy_data = pd.DataFrame({
    'id': cosmos_cat['id'], # Unique ID of the source
    'radius_sersic': cosmos_cat['radius_sersic'],
    'ra': cosmos_cat['ra'],
    'dec': cosmos_cat['dec'],
    'sersic': cosmos_cat['sersic'],
    'zpdf_med': cosmos_cat['zpdf_med'],
    'sfr_raw': cosmos_cat['sfr_raw'],
    'mass_raw': cosmos_cat['mass_raw'],
    'morphology': cosmos_cat['morphology'],
    'bovert': cosmos_cat['bovert'],
    'mass_CIGALE': cosmos_cat['mass_CIGALE'], 
    'sfr_CIGALE': cosmos_cat['sfr_CIGALE'],
    'log_radius_kpc': cosmos_cat['log_radius_kpc'],

    'time': all_time_total, # lbt [Myr]
    'time_err': all_time_err_total, # Error in lbt [Myr]
    'sfr': all_sfr_total, # Normalized SFR [1/yr]
    'sfr_err': all_sfr_err_total, # Normalized error in SFR [1/yr]
    'sfh_int': all_sfh_int_total, # Total integrated star formation history [M_sol]
    'sfh_int_err': all_sfh_int_err_total, # Error in sfh_integrated [M_sol]
    
    'zfinal': cosmos_cat['zfinal'],
    'mabs_nuv': cosmos_cat['mabs_nuv'],
    'mabs_j': cosmos_cat['mabs_r'],
    'mabs_r': cosmos_cat['mabs_j']
})





In [ ]:
print(galaxy_data['time']*10**(-3))

In [2]:
# 5. Store cleaned file
output_path = "C:\\Users\\usuario\\Documents\\TFG\\florah_training_SFR\\COSMOSWeb_Laura_processed_SFH.csv"

galaxy_data.to_csv(output_path, index=False)
print(f"Catálogo procesado guardado con {len(cosmos_cat)} filas.")

Catálogo procesado guardado con 588536 filas.


La última vez tardó unos 5min